# 01 · 索引链路观察（前半：切分）

本 Notebook 用来**肉眼观察**索引链路的中间产物（对应 spec §10 的观测点 1~2）。

下半部分（T3 完成后补）：向量的维度与数值。

> 对应任务：T2


In [ ]:
from pathlib import Path

from kbqa.chunking import split_text
from kbqa.config import settings
from kbqa.loaders import list_documents, load_document, save_upload

settings  # 打印出来，确认配置已从 .env 读入


## 1. 准备一篇文档

把 `SAMPLE` 改成你自己的 txt/md 文档路径（P0 前置条件要求你准备 2~3 篇）。
上传按内容哈希去重，重复运行不会产生多余文件。


In [ ]:
SAMPLE = Path("docs/spec.md")  # 换成你自己的文档路径

data_dir = Path("data")
doc = save_upload(SAMPLE.name, SAMPLE.read_bytes(), data_dir)
text = load_document(doc.path)

print("doc_id  :", doc.doc_id)
print("filename:", doc.filename)
print("字符数  :", len(text))
print("已上传文档:", [d.filename for d in list_documents(data_dir)])


## 2. 切分：看 chunk 总数与前 3 个 chunk 全文


In [ ]:
chunks = split_text(text, settings.chunk_size, settings.chunk_overlap)

print(f"chunk 总数 = {len(chunks)}")
print(f"chunk_size = {settings.chunk_size}, overlap = {settings.chunk_overlap}\n")

for i, (start, end, chunk_text) in enumerate(chunks[:3]):
    print(f"--- chunk[{i}]  start={start} end={end} 长度={end - start} ---")
    print(chunk_text)
    print()


## 3. 三条自检（T2 的验收标准）

1. 每个 chunk 长度 <= `chunk_size`
2. 相邻 chunk 恰好重叠 `chunk_overlap` 个字符
3. `text[start:end] == chunk_text` 逐条成立（位置不错位，引用才指得准）


In [ ]:
# 1) 长度不超上限
assert all(end - start <= settings.chunk_size for start, end, _ in chunks), "存在超长 chunk"

# 2) 相邻重叠恰好为 chunk_overlap
for (s1, e1, _), (s2, _, _) in zip(chunks, chunks[1:]):
    assert s2 == e1 - settings.chunk_overlap, f"重叠不对: {s1}-{e1} -> {s2}"

# 3) 位置与文本严格对齐
for start, end, chunk_text in chunks:
    assert text[start:end] == chunk_text, f"位置错位: ({start}, {end})"

print("三条自检全部通过")


## 4. 学习点：chunk_size 200 vs 800

chunk_size 越小 → chunk 越多、越碎；越大 → chunk 越少、每块噪声越多。
亲手跑一次，记一行结论（tasks.md 要求）。


In [ ]:
for size in (200, 800):
    print(f"chunk_size={size:>4} -> chunk 数 = {len(split_text(text, size, settings.chunk_overlap))}")


## 5. 向量化：观察向量的维度与数值（T3）

观测点（spec §10 索引第 3 步）：向量维度、L2 范数、前 5 维数值、分批次数。


In [ ]:
import math

from kbqa.embeddings import get_embedding_provider

embedder = get_embedding_provider(settings)
vecs = embedder.embed([t for _, _, t in chunks[:15]])  # 15 条，内部会自动分成 2 批

print("向量条数 :", len(vecs))
print("维度     :", len(vecs[0]))
print("L2 范数  :", round(math.sqrt(sum(x * x for x in vecs[0])), 6))
print("前 5 维  :", [round(x, 6) for x in vecs[0][:5]])


### 学习点：分批与顺序

15 条会被拆成 2 批（10 + 5），日志里能看到「批次数=2」。
顺序是这个模块最隐蔽的坑：错了**不会报错**，只会让检索结果整体错位。
下面用「把输入整体逆序再跑一遍」来验证顺序。


In [ ]:
def cos(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb)

texts = [f"这是第 {i} 条" for i in range(1, 16)]
v1 = embedder.embed(texts)
v2 = embedder.embed(list(reversed(texts)))
worst = min(cos(v2[i], v1[len(v1) - 1 - i]) for i in range(len(v1)))
print("逆序输入 vs 正序结果 的最差余弦 =", round(worst, 6), "(约等于 1 表示顺序没乱)")


In [ ]:
sim_rel = cos(embedder.embed(["如何切分文档"])[0], embedder.embed(["文本分块的方法"])[0])
sim_unrel = cos(embedder.embed(["如何切分文档"])[0], embedder.embed(["今天天气很好，适合出门散步"])[0])

print(f"语义相近 : {sim_rel:.4f}")
print(f"完全无关 : {sim_unrel:.4f}")
print("相近 > 无关 :", sim_rel > sim_unrel)


## 6. 入库与召回（T4）—— M1 里程碑

把前几步串成完整索引链路：切分 → 向量化 → 入库 → 召回。
重点观察 `stats()` 与落盘后的 `docs.json`。


In [ ]:
import json

import numpy as np

from kbqa.models import Chunk
from kbqa.vectorstore import VectorStore

chunk_objs = [
    Chunk(chunk_id=f"{doc.doc_id}-{i:04d}", doc_id=doc.doc_id, filename=doc.filename,
          index=i, text=t, start=s, end=e)
    for i, (s, e, t) in enumerate(chunks)
]

index_dir = Path("data/index")
store = VectorStore(index_dir)
store.remove_document(doc.doc_id)   # 重新索引 = 先删后加：重复运行不会累积重复 chunk
store.add(chunk_objs, np.asarray(embedder.embed([c.text for c in chunk_objs]), dtype=np.float32))
store.save()

print("stats =", store.stats())
print("docs.json =", json.loads((index_dir / "docs.json").read_text(encoding="utf-8")))


In [ ]:
q = np.asarray(embedder.embed(["如何切分文档"])[0], dtype=np.float32)

for r in store.search(q, top=5):
    print(f"#{r.rank + 1}  {r.chunk.chunk_id}  {r.score:.4f}  {r.chunk.text[:36]!r}")


至此索引链路完整跑通（M1）。下一步 T5 会给这 5 条召回结果做重排，
观察「谁被顶上来、谁被压下去」——那是本项目最关键的学习时刻之一。
